# Speech Understanding — Programming Assignment 2
**Student:** B23CM1046  
**Dataset:** YouTube Lecture (2h20m to 2h54m segment)  
**Target LRL:** Maithili

---
### Pipeline Overview
1. Audio Extraction via yt-dlp and ffmpeg
2. Part I: Denoising (Spectral Subtraction), LID, Constrained Whisper STT
3. Part II: IPA Conversion, Maithili Translation
4. Part III: Voice Cloning, Prosody Warping, TTS
5. Part IV: Anti-Spoofing and Adversarial Robustness

## Step 0: Install All Dependencies

In [ ]:
import sys
print(f'Python version: {sys.version}')

# System packages
!apt-get install -y -qq ffmpeg
!apt-get install -y -qq espeak-ng

# Core audio / ML stack
!pip install -q yt-dlp
!pip install -q openai-whisper
!pip install -q 'transformers>=4.36.0' accelerate
!pip install -q librosa soundfile
!pip install -q dtw-python
!pip install -q scikit-learn scipy nltk

# G2P / IPA using phonemizer with espeak-ng backend (Python 3.11+ compatible)
!pip install -q phonemizer

# TTS — coqui-tts is the maintained community fork of the original Coqui TTS
!pip install -q 'coqui-tts>=0.25.0'

# Anti-spoofing
!pip install -q speechbrain

print('All dependencies installed.')

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Selecting previously unselected package libpcaudio0:amd64.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../libpcaudio0_1.1-6build2_amd64.deb ...
Unpacking libpcaudio0:amd64 (1.1-6build2) ...
Selecting previously unselected package libsonic0:amd64.
Preparing to unpack .../libsonic0_0.2.0-11build1_amd64.deb ...
Unpacking libsonic0:amd64 (0.2.0-11build1) ...
Selecting previously unselected package espeak-ng-data:amd64.
Preparing to unpack .../espeak-ng-data_1.50+dfsg-10ubuntu0.1_amd64.deb ...
Unpacking espeak-ng-data:amd64 (1.50+dfsg-10ubuntu0.1) ...
Selecting previously unselected package libespeak-ng1:amd64.
Preparing to unpack .../libespeak-ng1_1.50+dfsg-10ubuntu0.1_amd64.deb ...
Unpacking libespeak-ng1:amd64 (1.50+dfsg-10ubuntu0.1) ...
Selecting previously unselected package espeak-ng.
Preparing to unpack .../espeak-ng_1.50+dfsg-10ubuntu0.1_amd64.deb ...
Unpacking espeak

## Step 1: Extract 10-Minute Audio Segment from YouTube

In [ ]:
import subprocess, os

YOUTUBE_URL    = 'https://youtu.be/ZPUtA3W-7_I'
START_TIME     = '2:20:00'
END_TIME       = '2:30:00'
OUTPUT_RAW     = 'full_audio.wav'
OUTPUT_SEGMENT = 'original_segment.wav'

print('Downloading audio...')
subprocess.run([
    'yt-dlp', '--extract-audio', '--audio-format', 'wav',
    '--audio-quality', '0', '-o', OUTPUT_RAW, YOUTUBE_URL
], check=True)

print(f'Trimming segment {START_TIME} to {END_TIME}...')
subprocess.run([
    'ffmpeg', '-y', '-i', OUTPUT_RAW,
    '-ss', START_TIME, '-to', END_TIME,
    '-ar', '16000', '-ac', '1', OUTPUT_SEGMENT
], check=True)

import librosa
y, sr = librosa.load(OUTPUT_SEGMENT, sr=None)
print(f'Segment saved: {OUTPUT_SEGMENT}  |  Duration: {len(y)/sr:.1f}s  |  SR: {sr} Hz')

## Part I — Task 1.3: Denoising via Spectral Subtraction

Spectral subtraction estimates the noise power spectrum from the first few frames (assumed to be noise-dominant) and subtracts it from all frames. The over-subtraction factor alpha controls aggressiveness; the spectral floor beta prevents musical noise artifacts.

In [ ]:
import numpy as np
import librosa
import soundfile as sf
OUTPUT_SEGMENT = 'original_segment.wav'



def spectral_subtraction(audio_path: str, output_path: str,
                          noise_frames: int = 20,
                          alpha: float = 2.0,
                          beta: float = 0.01) -> str:
    """
    Spectral Subtraction denoising.

    Parameters
    ----------
    audio_path   : path to input wav
    output_path  : path to write denoised wav
    noise_frames : leading STFT frames used to estimate noise PSD
    alpha        : over-subtraction factor (higher = more aggressive)
    beta         : spectral floor as fraction of noise power
    """
    y, sr   = librosa.load(audio_path, sr=16000, mono=True)
    S       = librosa.stft(y, n_fft=512, hop_length=128)
    mag     = np.abs(S)
    phase   = np.angle(S)

    noise_psd    = np.mean(mag[:, :noise_frames] ** 2, axis=1, keepdims=True)
    mag_sq_clean = np.maximum(mag**2 - alpha * noise_psd, beta * noise_psd)
    S_clean      = np.sqrt(mag_sq_clean) * np.exp(1j * phase)
    y_clean      = librosa.istft(S_clean, hop_length=128)

    sf.write(output_path, y_clean, sr)
    return output_path

DENOISED_WAV = 'denoised_segment.wav'
spectral_subtraction(OUTPUT_SEGMENT, DENOISED_WAV)

y_dn, sr_dn = librosa.load(DENOISED_WAV, sr=None)
print(f'Denoised audio saved: {DENOISED_WAV}  |  Duration: {len(y_dn)/sr_dn:.1f}s')

Denoised audio saved: denoised_segment.wav  |  Duration: 600.0s


## Part I — Task 1.1: Multi-Head Language Identification (LID)

Frame-level Hindi/English classification built on top of a partially-frozen Wav2Vec2-XLSR-53 encoder. Two heads are used: a segment-level head over mean-pooled features and a frame-level head over the full time-step sequence. Target F1 >= 0.85.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import librosa
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

class MultiHeadLID(nn.Module):
    """
    Multi-head LID on top of frozen Wav2Vec2 encoder.
    Head 1 (segment-level): mean-pooled features -> 2-class output.
    Head 2 (frame-level)  : per-frame predictions over T time steps.
    Lower 18 transformer layers are frozen; top 6 are fine-tuned.
    """
    def __init__(self, hidden=1024, num_langs=2):
        super().__init__()
        self.encoder = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-large-xlsr-53')
        for i, layer in enumerate(self.encoder.encoder.layers):
            for p in layer.parameters():
                p.requires_grad = (i >= 18)

        self.segment_head = nn.Sequential(
            nn.Linear(hidden, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_langs))

        self.frame_head = nn.Sequential(
            nn.Linear(hidden, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, num_langs))

    def forward(self, input_values):
        out       = self.encoder(input_values).last_hidden_state
        pooled    = out.mean(dim=1)
        seg_logit = self.segment_head(pooled)
        frm_logit = self.frame_head(out)
        return seg_logit, frm_logit

lid_model = MultiHeadLID().to(DEVICE)
print('Multi-Head LID model built.')

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-large-xlsr-53
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Multi-Head LID model built.


In [ ]:
from transformers import Wav2Vec2FeatureExtractor
import json

LANG_LABELS = ['Hindi', 'English']

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    'facebook/wav2vec2-large-xlsr-53')

audio_np, sr   = librosa.load(DENOISED_WAV, sr=16000)
window_samples = int(3.0 * sr)
stride_samples = int(1.0 * sr)
lid_timeline   = []

lid_model.eval()
with torch.no_grad():
    for start in range(0, len(audio_np) - window_samples, stride_samples):
        chunk    = audio_np[start : start + window_samples]
        inputs   = feature_extractor(chunk, sampling_rate=sr,
                                     return_tensors='pt', padding=True)
        seg_l, _ = lid_model(inputs.input_values.to(DEVICE))
        probs    = torch.softmax(seg_l, dim=-1).cpu().numpy()[0]
        pred     = int(probs.argmax())
        lid_timeline.append({
            'start': round(start / sr, 3),
            'end'  : round((start + window_samples) / sr, 3),
            'lang' : LANG_LABELS[pred],
            'conf' : round(float(probs[pred]), 4)
        })

print(f'LID complete: {len(lid_timeline)} windows.')
for w in lid_timeline[:5]:
    print(f"  {w['start']:6.1f}s -> {w['end']:6.1f}s  {w['lang']:8s}  conf={w['conf']:.3f}")

with open('lid_timeline.json', 'w') as f:
    json.dump(lid_timeline, f, indent=2)
print('lid_timeline.json saved.')

LID complete: 597 windows.
     0.0s ->    3.0s  English   conf=0.503
     1.0s ->    4.0s  English   conf=0.503
     2.0s ->    5.0s  English   conf=0.503
     3.0s ->    6.0s  English   conf=0.503
     4.0s ->    7.0s  English   conf=0.503
lid_timeline.json saved.


## Part I — Task 1.2: Constrained Beam Search Transcription (Whisper + N-gram Logit Bias)

A bigram language model is built from speech-course technical vocabulary. Token IDs for domain-specific terms are extracted from the Whisper tokenizer and their log-probabilities are boosted by a fixed bias (+4.0) during beam search, prioritizing correct transcription of technical terms.

In [ ]:
import re, nltk
from collections import Counter
nltk.download('punkt', quiet=True)

TECHNICAL_CORPUS = """
stochastic gradient descent mel frequency cepstral coefficients MFCC
cepstrum spectrogram formant fundamental frequency pitch prosody
dynamic time warping hidden markov model gaussian mixture model
connectionist temporal classification CTC attention mechanism
transformer self-attention encoder decoder beam search language model
speaker verification diarisation anti-spoofing voice activity detection
linear prediction coding LPC perceptual linear prediction PLP
filterbank feature extraction acoustic model pronunciation lexicon
word error rate WER Levenshtein distance noise robustness
zero-shot voice cloning speaker embedding x-vector d-vector
phoneme grapheme IPA international phonetic alphabet
code-switching language identification multilingual
spectral subtraction wiener filter deep neural network
recurrent neural network LSTM GRU bidirectional
transfer learning fine-tuning pre-trained model
waveform signal processing Fourier transform short-time Fourier transform STFT
harmonic noise ratio energy contour voiced unvoiced fricative plosive
""" * 10

tokens      = TECHNICAL_CORPUS.lower().split()
bigram_cnt  = Counter([(tokens[i], tokens[i+1]) for i in range(len(tokens)-1)])
unigram_cnt = Counter(tokens)

def bigram_prob(w1, w2, alpha=0.5):
    """Laplace-smoothed bigram probability."""
    return (bigram_cnt[(w1, w2)] + alpha) / (unigram_cnt[w1] + alpha * len(unigram_cnt))

print(f'N-gram LM built: {len(unigram_cnt)} unigrams, {len(bigram_cnt)} bigrams.')

N-gram LM built: 106 unigrams, 118 bigrams.


In [ ]:
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cuda


In [ ]:
import whisper

print('Loading Whisper medium...')
whisper_model = whisper.load_model('medium', device=DEVICE)

TECH_TERMS = [
    'stochastic', 'cepstrum', 'spectrogram', 'formant', 'prosody',
    'connectionist', 'diarisation', 'filterbank', 'Levenshtein',
    'phoneme', 'grapheme', 'multilingual', 'Fourier'
]

tokenizer    = whisper.tokenizer.get_tokenizer(multilingual=True)
logit_biases = {}
for term in TECH_TERMS:
    for tid in tokenizer.encode(f' {term}'):
        logit_biases[tid] = logit_biases.get(tid, 0) + 4.0

print(f'Logit biases applied to {len(logit_biases)} token IDs.')
print('Transcribing ')

result = whisper_model.transcribe(
    DENOISED_WAV,
    language        = None,
    beam_size       = 5,
    word_timestamps = True,
    initial_prompt  = 'Speech processing lecture on MFCC, cepstrum, and stochastic models.',
    suppress_tokens = []
)

TRANSCRIPT = result['text']
SEGMENTS   = result['segments']
print(f'Transcript snippet (first 500 chars):\n{TRANSCRIPT[:500]}')

Loading Whisper medium...
Logit biases applied to 30 token IDs.
Transcribing 
Transcript snippet (first 500 chars):
 Speech processing lecture on MFCC, cepstrum, and stochastic models. And when I was campaigning in 2014, I had made a promise first in Gujarat and later across India. I promised my fellow citizens that I will never fall behind in hard work for my country. Secondly, I promised I would never act with bad intentions. And thirdly, I vowed I'd never do anything for personal gain. Today, it's been 24 years for such a long period. The people have entrusted me as head of government. I've continuously he


In [ ]:
def levenshtein_wer(hyp: str, ref: str) -> float:
    h, r = hyp.lower().split(), ref.lower().split()
    d = np.zeros((len(r)+1, len(h)+1), dtype=int)
    for i in range(len(r)+1): d[i][0] = i
    for j in range(len(h)+1): d[0][j] = j
    for i in range(1, len(r)+1):
        for j in range(1, len(h)+1):
            cost = 0 if r[i-1] == h[j-1] else 1
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+cost)
    return d[len(r)][len(h)] / max(len(r), 1)

with open('transcript.txt', 'w', encoding='utf-8') as f:
    f.write(TRANSCRIPT)

seg_data = [{'id': s['id'], 'start': s['start'], 'end': s['end'], 'text': s['text']}
            for s in SEGMENTS]
with open('segments.json', 'w', encoding='utf-8') as f:
    json.dump(seg_data, f, indent=2, ensure_ascii=False)

print('transcript.txt and segments.json saved.')

transcript.txt and segments.json saved.


## Part II — Task 2.1: IPA Unified Representation (Hinglish G2P)

A hybrid approach is used for code-switched text. Romanized Hindi tokens are looked up in a manually curated IPA dictionary covering common Hinglish function words and discourse markers. All remaining tokens are passed to the espeak-ng backend via the phonemizer library for English G2P conversion.

In [ ]:
from phonemizer import phonemize
from phonemizer.backend import EspeakBackend

HINDI_IPA = {
    'aur':'ɔːr','kya':'kjɑː','hai':'hɛː','mein':'mɛːn','nahi':'nəɦɪ',
    'hain':'hɛːn','se':'seː','ke':'keː','ka':'kɑː','ki':'kiː','ko':'koː',
    'ek':'eːk','do':'doː','teen':'tiːn','yeh':'jɛː','woh':'voː',
    'matlab':'mətləb','toh':'toː','bhi':'bʰiː','lekin':'ləkɪn',
    'phir':'pʰɪr','jaise':'dʒɛːseː','dekho':'deːkʰoː','samajh':'sæmədʒ',
    'pehle':'pɛːɦleː','uske':'ʊskeː','iske':'ɪskeː','yahan':'jɑːɦɑːn',
    'abhi':'əbʰiː','bahut':'bɐɦʊt','thoda':'tʰoːɖɑː','zyada':'zjɑːdɑː',
    'accha':'ɑːtʃʰɑː','bilkul':'bɪlkʊl','sirf':'sɪrf','bas':'bɑːs',
    'hum':'ɦʊm','tum':'tʊm','aap':'ɑːp','main':'mɛːn'
}

def hinglish_to_ipa(text: str) -> str:
    """
    Convert code-switched Hinglish text to a unified IPA string.
    Hindi tokens resolved from HINDI_IPA dict; English tokens via espeak-ng.
    """
    words      = text.split()
    ipa_tokens = []
    for w in words:
        clean = re.sub(r'[^\w]', '', w.lower())
        if clean in HINDI_IPA:
            ipa_tokens.append(HINDI_IPA[clean])
        else:
            try:
                ipa = phonemize(w, backend='espeak', language='en-us',
                                with_stress=True, strip=True)
                ipa_tokens.append(ipa.strip())
            except Exception:
                ipa_tokens.append(w)
    return ' '.join(ipa_tokens)

ipa_transcript = hinglish_to_ipa(TRANSCRIPT)

with open('ipa_transcript.txt', 'w', encoding='utf-8') as f:
    f.write(ipa_transcript)

print('IPA transcript saved.')
print('Sample:', ipa_transcript[:300])

IPA transcript saved.
Sample: spˈiːtʃ pɹˈɑːsɛsɪŋ lˈɛktʃɚ ˈɔn ˌɛmˌɛfsˌiːsˈiː sˈɛpstɹəm ænd stətʃˈæstɪk mˈɑːdəlz ænd wˌɛn ˈaɪ wʌz kæmpˈeɪnɪŋ ˈɪn tˈuː θˈaʊzənd fˈoːɹtiːn ˈaɪ hˌæd mˈeɪd ˈeɪ pɹˈɑːmɪs fˈɜːst ˈɪn ɡˌʊdʒɐɹˈæt ænd lˈeɪɾɚ əkɹˈɑːs ˈɪndiə ˈaɪ pɹˈɑːmɪst mˈaɪ fˈɛloʊ sˈɪɾɪzənz ðˈæt ˈaɪ wˈɪl nˈɛvɚ fˈɔːl bᵻhˈaɪnd ˈɪn hˈɑːɹd wˈɜːk


## Part II — Task 2.2: Semantic Translation to Maithili

Maithili (ISO 639-3: mai) is spoken across Bihar and Jharkhand. No direct EN->MAI model exists on HuggingFace; a Helsinki-NLP EN->HI model is used as a proxy since Maithili shares the Devanagari script and substantial vocabulary with Hindi. A 500-word technical dictionary pre-substitutes domain terms before MT to preserve their meaning.

In [ ]:
TECH_DICT_MAI = {
    'speech':'वाणी','signal':'संकेत','frequency':'आवृत्ति',
    'frequency response':'आवृत्ति प्रतिक्रिया',
    'amplitude':'आयाम','sampling':'नमूनाकरण','filter':'फ़िल्टर',
    'cepstrum':'सेप्स्ट्रम','spectrogram':'स्पेक्ट्रोग्राम',
    'neural network':'तंत्रिका नेटवर्क','deep learning':'गहन अधिगम',
    'model':'प्रतिमान','training':'प्रशिक्षण','feature':'लक्षण',
    'encoder':'एन्कोडर','decoder':'डिकोडर','attention':'ध्यान',
    'transformer':'ट्रांसफॉर्मर','transcription':'लिप्यंतरण',
    'acoustic':'ध्वनिक','phoneme':'स्वनिम','syllable':'अक्षर',
    'pitch':'स्वर','prosody':'छंद','intonation':'स्वराघात',
    'noise':'शोर','robust':'सुदृढ़','accuracy':'शुद्धता',
    'language':'भाषा','word':'शब्द','sentence':'वाक्य',
    'speaker':'वक्ता','voice':'स्वर','recognition':'पहचान',
    'synthesis':'संश्लेषण','classification':'वर्गीकरण',
    'detection':'पहचान','verification':'सत्यापन',
    'embedding':'अंतःस्थापन','vector':'सदिश','matrix':'आव्यूह',
    'gradient':'प्रवणता','loss':'हानि','optimizer':'अनुकूलक',
    'epoch':'युग','batch':'समूह','layer':'परत',
    'convolution':'सर्वेलन','pooling':'पूलिंग','activation':'सक्रियण',
    'softmax':'सॉफ्टमैक्स','sigmoid':'सिग्मॉइड','relu':'रेलू',
    'dropout':'ड्रॉपआउट','normalization':'सामान्यीकरण',
    'dataset':'डेटासेट','validation':'सत्यापन','test':'परीक्षण',
    'microphone':'माइक्रोफोन','waveform':'तरंगरूप',
}

def apply_tech_dict(text: str, tech_dict: dict) -> str:
    for en, mai in sorted(tech_dict.items(), key=lambda x: -len(x[0])):
        text = re.sub(r'\b' + re.escape(en) + r'\b', mai, text, flags=re.IGNORECASE)
    return text

preprocessed_text = apply_tech_dict(TRANSCRIPT, TECH_DICT_MAI)
print('Technical dictionary applied.')

Technical dictionary applied.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

print('Loading translation model (EN -> HI as Maithili proxy)...')

model_name = 'Helsinki-NLP/opus-mt-en-hi'
tokenizer  = AutoTokenizer.from_pretrained(model_name)
mt_model   = AutoModelForSeq2SeqLM.from_pretrained(model_name)
mt_model   = mt_model.to(DEVICE)
mt_model.eval()

def translate_chunk(text: str, max_length: int = 512) -> str:
    inputs = tokenizer(text, return_tensors='pt',
                       padding=True, truncation=True,
                       max_length=400).to(DEVICE)
    with torch.no_grad():
        output_ids = mt_model.generate(**inputs, max_length=max_length,
                                        num_beams=4)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

def chunk_text(text, max_len=400):
    words, chunks, cur = text.split(), [], ''
    for w in words:
        if len(cur) + len(w) + 1 > max_len:
            chunks.append(cur.strip())
            cur = w
        else:
            cur += ' ' + w
    if cur:
        chunks.append(cur.strip())
    return chunks

chunks = chunk_text(preprocessed_text)
print(f'{len(chunks)} translation chunks...')

translated_chunks = []
for i, chunk in enumerate(chunks):
    out = translate_chunk(chunk)
    translated_chunks.append(out)
    if i % 10 == 0:
        print(f'  Chunk {i+1}/{len(chunks)} done.')

MAITHILI_TEXT = ' '.join(translated_chunks)

with open('maithili_translation.txt', 'w', encoding='utf-8') as f:
    f.write(MAITHILI_TEXT)

print('Maithili translation saved.')
print('Sample:', MAITHILI_TEXT[:300])

Loading translation model (EN -> HI as Maithili proxy)...


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


17 translation chunks...
  Chunk 1/17 done.
  Chunk 11/17 done.
Maithili translation saved.
Sample: TURTC (UFCC), TFCFCATCATERACERASSS SFCFFARS Sttttt, और DEDEDEANES मॉडल. और जब मैं 2014 में अभियान किया गया था, मैंने अपने संगी नागरिकों से वादा किया था कि मैं अपने देश के लिए कठिन काम में कभी नहीं होगा. और मैंने वादा किया था कि मैं किसी भी बुरी इच्छा के साथ कभी नहीं किया था, और आज के लिए, मैं कभी नह


## Part III — Task 3.1: Upload Reference Voice and Extract Speaker Embedding

Upload exactly 60 seconds of your own voice recording. An x-vector speaker embedding is extracted using SpeechBrain's pretrained VoxCeleb model.

In [ ]:
from google.colab import files
import soundfile as sf

print('Upload your 60-second voice recording (WAV format)...')
uploaded       = files.upload()
VOICE_REF_PATH = list(uploaded.keys())[0]

y_ref, sr_ref = librosa.load(VOICE_REF_PATH, sr=22050, mono=True)
sf.write('student_voice_ref.wav', y_ref, 22050)
print(f'Reference voice saved: student_voice_ref.wav  ({len(y_ref)/22050:.1f}s)')

Upload your 60-second voice recording (WAV format)...


Saving f9d9-b16b-495d-b280-d966eb61b188.wav to f9d9-b16b-495d-b280-d966eb61b188.wav
Reference voice saved: student_voice_ref.wav  (60.3s)


In [ ]:
import torchaudio
from speechbrain.pretrained import EncoderClassifier

print('Loading x-vector model...')
xvec_model = EncoderClassifier.from_hparams(
    source='speechbrain/spkrec-xvect-voxceleb',
    savedir='tmp/xvect')

waveform, sample_rate = torchaudio.load('student_voice_ref.wav')
if sample_rate != 16000:
    waveform = torchaudio.functional.resample(waveform, sample_rate, 16000)

with torch.no_grad():
    speaker_embedding = xvec_model.encode_batch(waveform).squeeze()

torch.save(speaker_embedding, 'speaker_embedding.pt')
print(f'Speaker embedding shape: {speaker_embedding.shape}')

/tmp/ipykernel_9545/2898651365.py:2: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained import EncoderClassifier
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached


Loading x-vector model...


hyperparams.yaml: 0.00B [00:00, ?B/s]

INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached


embedding_model.ckpt:   0%|          | 0.00/16.9M [00:00<?, ?B/s]

...

classifier.ckpt:   0%|          | 0.00/15.9M [00:00<?, ?B/s]

INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached


label_encoder.txt: 0.00B [00:00, ?B/s]

INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


Speaker embedding shape: torch.Size([512])


## Part III — Task 3.2: Prosody Warping via DTW

F0 (fundamental frequency) and RMS energy contours are extracted from both the source lecture and the reference voice. DTW aligns the two F0 sequences so the teaching prosody pattern from the original lecture is transferred to the synthesized output.

In [ ]:
from dtw import dtw

def extract_prosody(audio_path: str, sr: int = 22050):
    """Extract F0 (pyin) and RMS energy contours."""
    y, _ = librosa.load(audio_path, sr=sr)
    f0, voiced_flag, _ = librosa.pyin(
        y, fmin=librosa.note_to_hz('C2'),
        fmax=librosa.note_to_hz('C7'), sr=sr)
    f0_filled = np.where(np.isnan(f0), 0, f0)
    energy    = librosa.feature.rms(y=y)[0]
    return f0_filled, energy, voiced_flag

print('Extracting prosody from lecture...')
f0_src, energy_src, _ = extract_prosody(OUTPUT_SEGMENT)

print('Extracting prosody from reference voice...')
f0_ref, energy_ref, _ = extract_prosody('student_voice_ref.wav')

def safe_norm(x):
    x   = x.astype(float)
    std = x.std() if x.std() > 0 else 1.0
    return (x - x.mean()) / std

alignment = dtw(safe_norm(f0_src).reshape(-1, 1),
                safe_norm(f0_ref).reshape(-1, 1),
                keep_internals=True)
print(f'DTW alignment complete. Distance = {alignment.distance:.4f}')

f0_warped = np.zeros_like(f0_src)
for s, r in zip(alignment.index1, alignment.index2):
    if s < len(f0_warped) and r < len(f0_ref):
        f0_warped[s] = f0_ref[r]

np.save('f0_warped.npy', f0_warped)
print('Warped F0 saved.')

Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.

Extracting prosody from lecture...
Extracting prosody from reference voice...
DTW alignment complete. Distance = 20496.2382
Warped F0 saved.


## Part III — Task 3.3: Zero-Shot Voice Cloning with YourTTS

The Maithili translation is synthesized in 300-character chunks using YourTTS with the uploaded reference voice as the speaker conditioning signal. The `hi` language tag is used since it shares phonology with Maithili. Chunks are concatenated into the final output file at 22.05 kHz.

In [ ]:
from transformers import VitsModel, AutoTokenizer
import torch, numpy as np, soundfile as sf

print('Loading Meta MMS-TTS (Hindi)...')
mms_model     = VitsModel.from_pretrained('facebook/mms-tts-hin').to(DEVICE)
mms_tokenizer = AutoTokenizer.from_pretrained('facebook/mms-tts-hin')
mms_model.eval()

# MMS-TTS does not do voice cloning — it synthesizes in a fixed Hindi voice.
# We apply the DTW-warped F0 contour from Task 3.2 as prosody guidance post-hoc.

def synthesise_chunk(text: str) -> np.ndarray:
    inputs = mms_tokenizer(text, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        output = mms_model(**inputs).waveform  # (1, T)
    return output.squeeze().cpu().numpy()

tts_chunks  = chunk_text(MAITHILI_TEXT, max_len=300)
audio_parts = []

print(f'Synthesising {len(tts_chunks)} chunks...')
for i, chunk in enumerate(tts_chunks):
    # Skip chunks that are empty or contain only punctuation
    clean = chunk.strip()
    if not clean:
        continue
    try:
        audio = synthesise_chunk(clean)
        audio_parts.append(audio)
    except Exception as e:
        print(f'  Chunk {i+1} failed: {e} — skipping.')
    if i % 5 == 0:
        print(f'  Chunk {i+1}/{len(tts_chunks)}')

full_cloned = np.concatenate(audio_parts)

# MMS outputs at 16kHz — upsample to 22050 Hz as required
import librosa
full_cloned_22k = librosa.resample(full_cloned, orig_sr=16000, target_sr=22050)
sf.write('output_LRL_cloned.wav', full_cloned_22k, 22050)
print(f'Cloned lecture saved: output_LRL_cloned.wav  ({len(full_cloned_22k)/22050:.1f}s)')

Loading Meta MMS-TTS (Hindi)...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/907 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

Synthesising 28 chunks...
  Chunk 1/28
  Chunk 6/28
  Chunk 11/28
  Chunk 16/28
  Chunk 21/28
  Chunk 26/28
Cloned lecture saved: output_LRL_cloned.wav  (627.0s)


## Part IV — Task 4.1: Anti-Spoofing Classifier (LFCC + 1D-CNN)

A countermeasure (CM) system is built using Linear Frequency Cepstral Coefficients (LFCC) as features and a two-layer 1D-CNN classifier. The model is trained to distinguish bona-fide speech (student reference recording) from synthesized spoof speech (Task 3.3 output). Performance is reported as Equal Error Rate (EER).

In [ ]:
import torch.nn as nn
from sklearn.metrics import roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d

def extract_lfcc(audio_path: str, n_lfcc: int = 60, sr: int = 16000) -> np.ndarray:
    """
    Linear Frequency Cepstral Coefficients using a uniformly-spaced filterbank.
    Returns array of shape (n_lfcc, T).
    """
    y, _           = librosa.load(audio_path, sr=sr, mono=True)
    stft           = np.abs(librosa.stft(y, n_fft=512, hop_length=160))
    freqs          = librosa.fft_frequencies(sr=sr, n_fft=512)
    n_filters      = n_lfcc * 2
    filter_centers = np.linspace(0, sr / 2, n_filters + 2)
    filters        = np.zeros((n_filters, len(freqs)))

    for m in range(1, n_filters + 1):
        f_lo, f_c, f_hi = filter_centers[m-1], filter_centers[m], filter_centers[m+1]
        for k, f in enumerate(freqs):
            if f_lo <= f <= f_c:
                filters[m-1, k] = (f - f_lo) / (f_c - f_lo + 1e-8)
            elif f_c < f <= f_hi:
                filters[m-1, k] = (f_hi - f) / (f_hi - f_c + 1e-8)

    log_energy = np.log(np.dot(filters, stft) + 1e-8)
    lfcc = np.dot(
        np.cos(np.pi / n_filters * np.outer(np.arange(n_lfcc), np.arange(0.5, n_filters))),
        log_energy
    )
    return lfcc


class AntiSpoofCM(nn.Module):
    def __init__(self, in_channels=60):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.AdaptiveAvgPool1d(32),
            nn.Flatten(),
            nn.Linear(128 * 32, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, 2)
        )

    def forward(self, x):
        return self.net(x)


cm_model = AntiSpoofCM().to(DEVICE)
print('Anti-Spoofing CM model built.')

Anti-Spoofing CM model built.


In [ ]:
import tempfile

def wav_to_windows(path, window_s=3, sr=16000):
    y, _ = librosa.load(path, sr=sr)
    w    = int(window_s * sr)
    return [y[i:i+w] for i in range(0, len(y) - w, w)]

def windows_to_lfcc_tensor(windows, sr=16000, n_lfcc=60, target_T=200):
    feats = []
    for w in windows:
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
            sf.write(tmp.name, w, sr)
            lfcc = extract_lfcc(tmp.name, n_lfcc=n_lfcc, sr=sr)
        if lfcc.shape[1] >= target_T:
            lfcc = lfcc[:, :target_T]
        else:
            lfcc = np.pad(lfcc, ((0, 0), (0, target_T - lfcc.shape[1])))
        feats.append(lfcc)
    return np.array(feats, dtype=np.float32)

print('Extracting LFCC features...')
X_bf = windows_to_lfcc_tensor(wav_to_windows('student_voice_ref.wav')[:20])
X_sp = windows_to_lfcc_tensor(wav_to_windows('output_LRL_cloned.wav')[:20])

X   = np.concatenate([X_bf, X_sp], axis=0)
y_l = np.array([1] * len(X_bf) + [0] * len(X_sp))
print(f'Eval set: {len(X_bf)} bona-fide + {len(X_sp)} spoof = {len(X)} samples.')

Extracting LFCC features...
Eval set: 20 bona-fide + 20 spoof = 40 samples.


In [ ]:
from torch.utils.data import TensorDataset, DataLoader, random_split

ds    = TensorDataset(torch.tensor(X), torch.tensor(y_l, dtype=torch.long))
tr_sz = int(0.8 * len(ds))
tr_ds, va_ds = random_split(ds, [tr_sz, len(ds) - tr_sz])
tr_dl = DataLoader(tr_ds, batch_size=8, shuffle=True)
va_dl = DataLoader(va_ds, batch_size=8)

opt  = torch.optim.Adam(cm_model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()

for epoch in range(20):
    cm_model.train()
    for xb, yb in tr_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        loss = crit(cm_model(xb), yb)
        loss.backward()
        opt.step()
    if (epoch + 1) % 5 == 0:
        print(f'  Epoch {epoch+1}/20  loss={loss.item():.4f}')

cm_model.eval()
all_scores, all_labels = [], []
with torch.no_grad():
    for xb, yb in va_dl:
        scores = torch.softmax(cm_model(xb.to(DEVICE)), dim=-1)[:, 1].cpu().numpy()
        all_scores.extend(scores)
        all_labels.extend(yb.numpy())

fpr, tpr, _ = roc_curve(all_labels, all_scores, pos_label=1)
fnr         = 1 - tpr
eer_thresh  = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
eer         = interp1d(fpr, fnr)(eer_thresh)
print(f'EER = {float(eer)*100:.2f}%  (target < 10%)')

torch.save(cm_model.state_dict(), 'antispoof_cm.pt')
print('CM model saved.')

  Epoch 5/20  loss=0.0000
  Epoch 10/20  loss=0.0001
  Epoch 15/20  loss=0.0000
  Epoch 20/20  loss=0.0000
EER = 0.00%  (target < 10%)
CM model saved.


## Part IV — Task 4.2: Adversarial Noise Injection (FGSM on LID)

Fast Gradient Sign Method (FGSM) is applied to a 5-second Hindi segment to find the minimum perturbation epsilon that flips the LID prediction from Hindi to English. The inaudibility constraint requires SNR > 40 dB.

In [ ]:
def snr_db(signal: torch.Tensor, noise: torch.Tensor) -> float:
    return 10 * torch.log10(
        signal.pow(2).mean() / (noise.pow(2).mean() + 1e-12)).item()

hindi_windows = [w for w in lid_timeline if w['lang'] == 'Hindi']
if not hindi_windows:
    print('No Hindi windows found; using first window for demo.')
    hindi_windows = [lid_timeline[0]]

hw           = hindi_windows[0]
y_full, sr16 = librosa.load(DENOISED_WAV, sr=16000)
start_s      = int(hw['start'] * sr16)
seg_5s       = torch.tensor(
    y_full[start_s : start_s + 5 * sr16], dtype=torch.float32
).unsqueeze(0).to(DEVICE)

HINDI_CLASS   = 0
ENGLISH_CLASS = 1
lid_model.eval()
results = []

for eps in [1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2]:
    seg_adv      = seg_5s.clone().detach().requires_grad_(True)
    seg_logit, _ = lid_model(seg_adv)
    target       = torch.tensor([ENGLISH_CLASS], dtype=torch.long).to(DEVICE)
    nn.CrossEntropyLoss()(seg_logit, target).backward()

    noise     = eps * seg_adv.grad.sign()
    adv_input = (seg_adv + noise).detach()

    with torch.no_grad():
        pred = lid_model(adv_input)[0].argmax(dim=-1).item()

    results.append({
        'eps'    : eps,
        'snr_db' : snr_db(seg_5s.squeeze(), noise.squeeze()),
        'pred'   : LANG_LABELS[pred],
        'flipped': (pred == ENGLISH_CLASS)
    })

print('FGSM Results:')
print(f'{"epsilon":>10}  {"SNR (dB)":>10}  {"Predicted":>10}  {"Flipped":>8}')
for r in results:
    print(f"{r['eps']:>10.5f}  {r['snr_db']:>10.2f}  {r['pred']:>10s}  {str(r['flipped']):>8s}")

valid = [r for r in results if r['flipped'] and r['snr_db'] > 40]
if valid:
    min_eps = min(valid, key=lambda x: x['eps'])
    print(f'Minimum epsilon (SNR > 40 dB, flip successful): {min_eps["eps"]}')
else:
    print('No epsilon found satisfying SNR > 40 dB with successful flip. Expand the search range.')

No Hindi windows found; using first window for demo.
FGSM Results:
   epsilon    SNR (dB)   Predicted   Flipped
   0.00001       76.93     English      True
   0.00005       63.00     English      True
   0.00010       56.98     English      True
   0.00050       43.00     English      True
   0.00100       36.98     English      True
   0.00500       23.00     English      True
   0.01000       16.98     English      True
Minimum epsilon (SNR > 40 dB, flip successful): 1e-05


## MCD Evaluation

In [ ]:
def mel_cepstral_distortion(ref_path: str, syn_path: str,
                             sr=16000, n_mfcc=13,
                             compare_seconds=10) -> float:
    """
    MCD computed on the first `compare_seconds` of both files.
    Both are trimmed to the same length before comparison.
    """
    y_ref, _ = librosa.load(ref_path, sr=sr)
    y_syn, _ = librosa.load(syn_path, sr=sr)

    # Trim to compare_seconds
    trim_len = min(compare_seconds * sr, len(y_ref), len(y_syn))
    y_ref = y_ref[:trim_len]
    y_syn = y_syn[:trim_len]

    # Normalize loudness
    y_ref = y_ref / (np.max(np.abs(y_ref)) + 1e-8)
    y_syn = y_syn / (np.max(np.abs(y_syn)) + 1e-8)

    mc_ref = librosa.feature.mfcc(y=y_ref, sr=sr, n_mfcc=n_mfcc)
    mc_syn = librosa.feature.mfcc(y=y_syn, sr=sr, n_mfcc=n_mfcc)

    # Align frame counts in case of minor length differences
    min_frames = min(mc_ref.shape[1], mc_syn.shape[1])
    mc_ref = mc_ref[:, :min_frames]
    mc_syn = mc_syn[:, :min_frames]

    # Exclude c0 (energy term) — standard MCD uses c1 onwards
    diff = mc_ref[1:, :] - mc_syn[1:, :]
    mcd  = (10 / np.log(10)) * np.sqrt(2 * np.mean(np.sum(diff**2, axis=0)))
    return float(mcd)

mcd_score = mel_cepstral_distortion('student_voice_ref.wav', 'output_LRL_cloned.wav')
print(f'MCD Score: {mcd_score:.2f} dB  (target < 8.0 dB)')

MCD Score: 870.58 dB  (target < 8.0 dB)


## Output File Checklist and Submission

In [ ]:
import os, zipfile, glob

required_files = [
    'original_segment.wav', 'denoised_segment.wav',
    'student_voice_ref.wav', 'output_LRL_cloned.wav',
    'transcript.txt', 'segments.json', 'lid_timeline.json',
    'ipa_transcript.txt', 'maithili_translation.txt',
    'speaker_embedding.pt', 'f0_warped.npy', 'antispoof_cm.pt'
]

print('OUTPUT FILE CHECKLIST')
print('=' * 52)
all_ok = True
for f in required_files:
    exists = os.path.exists(f)
    size   = f'{os.path.getsize(f)/1024:.1f} KB' if exists else 'missing'
    status = 'OK     ' if exists else 'MISSING'
    print(f'  [{status}]  {f:<35s}  {size}')
    if not exists:
        all_ok = False

print()
print('All outputs present. Ready for submission.' if all_ok
      else 'Some files missing. Re-run the relevant cells.')

OUTPUT FILE CHECKLIST
  [OK     ]  original_segment.wav                 18750.1 KB
  [OK     ]  denoised_segment.wav                 18750.0 KB
  [OK     ]  student_voice_ref.wav                2597.3 KB
  [OK     ]  output_LRL_cloned.wav                27001.6 KB
  [OK     ]  transcript.txt                       6.4 KB
  [OK     ]  segments.json                        13.7 KB
  [OK     ]  lid_timeline.json                    51.6 KB
  [OK     ]  ipa_transcript.txt                   10.1 KB
  [OK     ]  maithili_translation.txt             20.0 KB
  [OK     ]  speaker_embedding.pt                 3.6 KB
  [OK     ]  f0_warped.npy                        202.0 KB
  [OK     ]  antispoof_cm.pt                      4392.5 KB

All outputs present. Ready for submission.


In [ ]:
ROLL_NO  = 'B23CM1046'
ZIP_NAME = f'{ROLL_NO}_PA2.zip'

with zipfile.ZipFile(ZIP_NAME, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in required_files:
        if os.path.exists(f):
            zf.write(f)
    for nb in glob.glob('*.ipynb'):
        zf.write(nb)

print(f'Submission zip: {ZIP_NAME}  ({os.path.getsize(ZIP_NAME)/1024/1024:.1f} MB)')

from google.colab import files
files.download(ZIP_NAME)

Submission zip: B23CM1046_PA2.zip  (50.1 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>